In [1]:
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer
import torch
import pandas as pd
from tqdm import tqdm

/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using device:", device)

Using device: mps


In [ ]:
model = AutoPeftModelForCausalLM.from_pretrained(
    "pykale/llama-2-13b-ocr",
    token='fill_my_token',
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,   # MPS works well with float16/bfloat16
)
model.to(device)

tokenizer = AutoTokenizer.from_pretrained("pykale/llama-2-13b-ocr")

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  5.50it/s]
/Users/mstudio/miniforge3/envs/kkk/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


In [27]:
data=pd.read_feather('/Volumes/T7/chroniclingamerica/kkk/revival_kkk_all')

In [28]:
data

,city,state,date,lCCN,text,title
0,minneapolis,minnesota,1923-02-08,['2019271211'],VCa N uY i 0 Wiiv m voice KNIGHTS of the KU KL...,voice of the knights of the ku klux klan (minn...
1,aberdeen,mississippi,1921-01-28,['sn86074011'],jfllE AEECQEEIi WEEKLY V 77 TY tfYt W 7 I EVER...,"the aberdeen weekly (aberdeen, miss.) 1878-1933"
2,philadelphia,pennsylvania,1921-09-20,['sn83045211'],IPWPJIIWP JvaajMiill II IIIHIRWWMPMMI ft V i V...,evening public ledger (philadelphia [pa.]) 191...
3,indianapolis,indiana,1924-10-30,['sn82015313'],THLKfcSJJAY OCT 301024 KLAN GIVES SLATE IN CON...,the indianapolis times (indianapolis [ind.]) 1...
4,philadelphia,pennsylvania,1921-09-17,['sn83045211'],r i i j lv 3T i t EVENING PUBLIC LEDERPfilLADE...,evening public ledger (philadelphia [pa.]) 191...
...,...,...,...,...,...,...
155,indianapolis,indiana,1925-11-13,['sn82015313'],Home Edition TIE Times weekly American Legion ...,the indianapolis times (indianapolis [ind.]) 1...
156,indianapolis,indiana,1928-11-01,['sn82015313'],Second Section 15 WILL FACE COURT IN AUTO THEF...,the indianapolis times (indianapolis [ind.]) 1...
157,west union,ohio,1923-09-13,['sn83035189'],THE PEOPLES DEFENDER VOLUME LVIII WEST UNION O...,"the people's defender (west union, adams count..."
158,indianapolis,indiana,1926-10-11,['sn82015313'],Home Edition You Have to Iload The Times If Yo...,the indianapolis times (indianapolis [ind.]) 1...


In [ ]:
ocr = "The defendant wits'fined �5 and costs."

prompt = f"""### Instruction:
Fix the OCR errors in the provided text.

### Input:
{ocr}

### Response:
"""

In [ ]:
inputs = tokenizer(
    prompt,
    max_length=1024,
    return_tensors='pt',
    truncation=True
)

input_ids = inputs.input_ids.to(device)

with torch.inference_mode():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.1,
        top_k=40
    )

In [19]:
' '.join(data['context'][4])

'in the mont gomery alabama journal in last weeks issue of the digest for in stance louisiana members of the klan were charged by louisiana editors with the murder of two young men of mer kouge named daniel and rich'

In [21]:
prompt

'### Instruction:\n  Fix the OCR errors in the provided text.\n\n  ### Input:\n  our american flag should know the truth concerning the principles activities and ideals of the knights of the ku klux klan north star klan no 2 of minneapolis have caused this sheet to be issued a defence of the ku\n\n  ### Response:\n  '

In [ ]:
inputs = tokenizer(
    prompt,
    max_length=2048,
    return_tensors='pt',
    truncation=True
)

input_ids = inputs.input_ids.to(device)

with torch.inference_mode():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.1,
        top_k=40
    )

pred = tokenizer.batch_decode(outputs.cpu().numpy(), skip_special_tokens=True)[0][len(prompt):].strip()

print(pred)

In [37]:
data['text'].iloc[2]

'IPWPJIIWP JvaajMiill II IIIHIRWWMPMMI ft V i Vf V t ki frl y iOUf mm MI Am Hi fflWMtSEMPEROR EULOGIZES DISORDER AftD feiil MAGISTRATES RAP Ink AAiniT Ar TUP ll ll rJflKIIUMIItMUA Ministers and Others Join in t Praiso of Evoning Publio Ledger for Expos lOTRY IS DENOUNCED Honest Courage of Expose Praised by Business Men To thu Editor ot Kutnlnp Public Ieiqrr Sir It nlwnyn was a known fnct thnt the mnrnltiR Pcmtc TKnonn utood for Juaticr tirnl rWitpmiMiem and It teem thnt Its ofNiirins the Evieino Pinuc Lepokr is fol lowing tlie footsteps of tlie fnther publication At n mcotlnc of the South Street Buitnesi Men Association held Thursday September 15 It wns Ilnnnlmoitslr revived thnt n vote of thanks be extended to your vnlun ble paper for It honest eonrnce and conviction In cxponltiK the Ku Klix Kln n ronglomerntlon of hoodlum and swindlers masquerading ns patriotic Amerlrans IIoplnB that you may carry on the food work of exposing the sorailed True Americans II M LEVY President S VRAM Vic

In [32]:
ocr_correction

["0 VOLUME XL NUMBER 103 KNIGHTS of the KU KLUX KLAN.\n0 THREE Propagandists of Invisible Empire Turn Guns on Each Other.\n0 ALEXANDRIA MAKES GREAT RECORD IN HEALTH CONDITIONS.\n0 AMARILLO DAILY NEWS VOL XIII NO 201 ACCUSED PREACHERS ARE INDICTED.\n0 KLAN AND LIQUOR OHIO'S BIG ISSUES Seven-Cornered Fight for Governorship.\n0 POPPIES MADE BY CHILDREN TO HONOR HEARST By the Associated Press.\n0 VOLUME HI NUMBER 63 SEWARD, ALASKA, FRIDAY, MARCH 15, 1912.\n0 HAUTE 4 5 VOLUME 69 CARSON CITY, NEVADA, THURSDAY, MARCH 15, 1912.\n0 SPORTS NEWS Griffs Must Fight to Hold Fifth Place.\n0 JURORS CHOSEN FOR SPRING TERM Judge Peters Discharges 100.",
 '1. THE AESTHETIC WEEKLY VOLUME 1. TY THE WEEK. EVER.',
 '2 MONDAY JANUARY 17 1921 ORDERNEGROES TO QUIT DIXIE.',
 '3 THURSDAY OCTOBER 301924 KLAN GIVES SLATE IN CONGRESS.\n3 Friday April 41924 WASHINGTON BUREAU.\n3 MORE PUZZLE FOR THE CLAN Philip E Fox publicly denies that he is a member of the Ku Klux Klan.\n3 VETERANS MUST WAIT FOR PAY NOT SUFFICIENT 

In [29]:
ocr_correction=[]
for i in tqdm(range(len(data['text']))):
  # ocr_context=' '.join(data['text'][i]) 
  ocr_context=data['text'][i]
  prompt = f"""### Instruction:
  Fix the OCR errors in the provided text.

  ### Input:
  {ocr_context}

  ### Response:
  """
  inputs = tokenizer(
    prompt,
    max_length=2048,
    return_tensors='pt',
    truncation=True
  )
  input_ids = inputs.input_ids.to(device)
  with torch.inference_mode():
        outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.1,
        top_k=40
    )
  prediction = tokenizer.batch_decode(outputs.cpu().numpy(), skip_special_tokens=True)[0][len(prompt):].strip()
  ocr_correction.append(prediction)

  0%|          | 14/53614 [01:59<127:02:50,  8.53s/it]


KeyboardInterrupt: 

In [ ]:
ocr_correction=[]
for ocr in tqdm(data['article'].to_numpy()):
  prompt = f"""### Instruction:
  Fix the OCR errors in the provided text.

  ### Input:
  {ocr}

  ### Response:
  """
  input_ids = tokenizer(prompt, max_length=1024, return_tensors='pt', truncation=True)
  with torch.inference_mode():
    outputs = model.generate(input_ids=input_ids, max_new_tokens=1024, do_sample=True, temperature=0.7, top_p=0.1, top_k=40)
  prediction = tokenizer.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0][len(prompt):].strip()
  ocr_correction.append(prediction)

In [6]:
prompt

'### Instruction:\n  Fix the OCR errors in the provided text.\n\n  ### Input:\n  VCa N uY i 0 Wiiv m voice KNIGHTS of the KU KLUX KL AN Volume I Believing that all people living under Our American Flag should Know the Truth concerning the Principles Activities and Ideals of the Knights of the Ku Klux Klan North Star Klan No 2 of Minneapolis have caused this sheet to be issued A DEFENCE OF THE KU KLUX KLAN Reprint from Literary Digest Jan 20 1923 The Ku Klux Klan has been charged with all sorts of crimes and mis demeanors but so far not one charge has been proved we read in the Mont gomery Alabama Journal In last weeks issue of The Digest for in stance Louisiana members of the Klan were charged by Louisiana editors with the murder of two young men of Mer Kouge named Daniel and Rich ards and while there was denial on the part of Ku Klux officials that this horrible crime had been committed by members of the Klan there was no specific defense of the murderers in particular or the Klan in 

In [22]:
inputs = tokenizer(
    prompt,
    max_length=2048,
    return_tensors='pt',
    truncation=True
)

input_ids = inputs.input_ids.to(device)

with torch.inference_mode():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.1,
        top_k=40
    )

pred = tokenizer.batch_decode(outputs.cpu().numpy(), skip_special_tokens=True)[0][len(prompt):].strip()

print(pred)

1. That the American flag should know the truth concerning the principles, activities, and ideals of the Knights of the Ku Klux Klan, North Star Klan No. 2, of Minneapolis, have caused this sheet to be issued a defence of the ku


In [23]:
ocr_context

'our american flag should know the truth concerning the principles activities and ideals of the knights of the ku klux klan north star klan no 2 of minneapolis have caused this sheet to be issued a defence of the ku'